# Phase 5: Fine-Tuning Legal-BERT

This notebook fine-tunes the `nlpaueb/legal-bert-base-uncased` model using Sentence Transformers with `MultipleNegativesRankingLoss`.

### Instructions:
1. Enable **GPU** (Runtime -> Change runtime type -> T4 GPU).
2. Upload `chunk_pairs.parquet` and `chunks.parquet` from `data/processed/` when prompted.

In [ ]:
!pip install sentence-transformers pandas pyarrow

In [ ]:
import json
import pandas as pd
from sentence_transformers import SentenceTransformer, models, InputExample, losses
from torch.utils.data import DataLoader
from google.colab import files
import os

# 1. Upload Data
print("Please upload 'chunk_pairs.parquet' and 'chunks.parquet':")
uploaded = files.upload()

# 2. Load Data
print("Loading data...")
df_pairs = pd.read_parquet('chunk_pairs.parquet')
df_chunks = pd.read_parquet('chunks.parquet')

# Map IDs to text
id_to_text = dict(zip(df_chunks['chunk_id'], df_chunks['text']))

train_examples = []
for _, row in df_pairs.iterrows():
    q_text = id_to_text.get(row['query_chunk_id'])
    p_text = id_to_text.get(row['positive_chunk_id'])
    if q_text and p_text:
        train_examples.append(InputExample(texts=[q_text, p_text]))

print(f"Prepared {len(train_examples)} training examples.")

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

# 3. Model Setup
model_name = 'nlpaueb/legal-bert-base-uncased'
word_embedding_model = models.Transformer(model_name, max_seq_length=512)
pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), pooling_mode='mean')
model = SentenceTransformer(modules=[word_embedding_model, pooling_model])

train_loss = losses.MultipleNegativesRankingLoss(model)

# 4. Fine-Tuning
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3,
    warmup_steps=int(len(train_dataloader) * 3 * 0.1), # 10% warmup ratio
    optimizer_params={'lr': 2e-5},
    weight_decay=0.01,
    use_amp=True # fp16
)

# 5. Save & Download
output_dir = 'legalbert-finetuned'
model.save(output_dir)

!zip -r legalbert-finetuned.zip legalbert-finetuned
files.download('legalbert-finetuned.zip')